In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import torch

from src.model import UNet

device = torch.device("cpu")

model = UNet().to(device)

print(model)

UNet(
  (enc1): DoubleConv(
    (block): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): DoubleConv(
    (block): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplace=True)
  

In [4]:
num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"{num_params:,} trainable parameters")

31,036,481 trainable parameters


In [5]:
from torch.utils.data import DataLoader
from src.dataset import OilSpillDataset


train_dataset = OilSpillDataset(
    image_dir="../data/images/train",
    mask_dir="../data/masks/train",
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

images, masks = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():
    outputs = model(images)

print("input :", images.shape)
print("output:", outputs.shape)

/opt/homebrew/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


input : torch.Size([16, 1, 256, 256])
output: torch.Size([16, 1, 256, 256])


In [6]:
print(torch.backends.mps.is_available())

True


MPS should help reduce the training times here, considering Apple Silicon.

In [7]:
from src.losses import DiceBCELoss

criterion = DiceBCELoss()

outputs = model(images)

loss = criterion(
    outputs,
    masks.to(device)
)

print("loss:", loss.item())

loss: 1.4593708515167236
